# Data

In [1]:
import json

with open('data/result.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [2]:
import pandas as pd

df = pd.DataFrame(data['messages'])

In [3]:
from utils import preprocess_df

df = preprocess_df(df)

Загружено сообщений для анализа: 32738


# Pipeline

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

In [6]:
from sklearn.cluster import DBSCAN

clusterer = DBSCAN(eps=0.01, min_samples=3)

In [7]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=model,
    hdbscan_model=clusterer,
    umap_model=reducer,
    verbose=True
)

topics, probs = topic_model.fit_transform(df['clean_text'].tolist())

2026-03-29 12:01:17,304 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/1024 [00:00<?, ?it/s]

2026-03-29 12:03:45,263 - BERTopic - Embedding - Completed ✓
2026-03-29 12:03:45,264 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 12:04:49,790 - BERTopic - Dimensionality - Completed ✓
2026-03-29 12:04:49,792 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 12:04:50,218 - BERTopic - Cluster - Completed ✓
2026-03-29 12:04:50,234 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 12:04:50,864 - BERTopic - Representation - Completed ✓


# Analysis

In [8]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25196,-1_не_меня_ты_что,"[не, меня, ты, что, ну, просто, мне, на, бля, ...","[Это мне надо, Вот так бля, бля ребят играл и ..."
1,0,117,0_курить_курит_сигареты_курю,"[курить, курит, сигареты, курю, куришь, курил,...","[курить хочу, Я курить, Курить]"
2,1,74,1_глеб_хлеба_батон_хохлочел,"[глеб, хлеба, батон, хохлочел, тупица, спокойн...","[Глеб кто бля, Это глеб, Глеб???]"
3,2,55,2_брат_братски_братья_братух,"[брат, братски, братья, братух, ахахахах, конф...","[По братски, С братиками конфликтики порешать,..."
4,3,51,3_gyrozeppeli2_попали_рест_открыла,"[gyrozeppeli2, попали, рест, открыла, тусит, п...","[@Gyrozeppeli2, @Gyrozeppeli2, @Gyrozeppeli2]"
...,...,...,...,...,...
949,948,3,948_шнейне___,"[шнейне, , , , , , , , , ]","[Шнейне, Шнейне, Шнейне]"
950,949,3,949_препарат_втираю_юзаю_мажет,"[препарат, втираю, юзаю, мажет, меня, , , , , ]","[меня мажет препарат, Я юзаю препарат, Я втира..."
951,950,3,950_двоек_пропусков_нулевой_сообщений,"[двоек, пропусков, нулевой, сообщений, ноль, д...","[Ноль сообщений видел, с нулевой, 0 пропусков ..."
952,951,3,951_хуже_второй_первый_чем,"[хуже, второй, первый, чем, есть, же, но, да, , ]","[Но второй хуже да, Хуже чем первый же?, есть ..."


In [9]:
topic_model.get_topic_info().describe()

,Topic,Count
count,954.000000,954.000000
mean,475.500000,34.316562
std,275.540378,815.539217
min,-1.000000,3.000000
25%,237.250000,3.000000
50%,475.500000,5.000000
75%,713.750000,9.000000
max,952.000000,25196.000000
